## 1-minute introduction to Jupyter ##

A Jupyter notebook consists of cells. Each cell contains either text or code.

A text cell will not have any text to the left of the cell. A code cell has `In [ ]:` to the left of the cell.

If the cell contains code, you can edit it. Press <kbd>Enter</kbd> to edit the selected cell. While editing the code, press <kbd>Enter</kbd> to create a new line, or <kbd>Shift</kbd>+<kbd>Enter</kbd> to run the code. If you are not editing the code, select a cell and press <kbd>Ctrl</kbd>+<kbd>Enter</kbd> to run the code.

---

# Object-Oriented Programming

This lesson continues from the previous chapter on Encapsulation. In that lesson, we created a `Grid` class with a public interface, including `get()` and `set()` methods. The public **interface** hides the **implementation** details of the class, allowing the implementation code to be refactored without affecting other code that relies on the public interface.

Yet one problem remains. We actually have two kinds of boards: a targetting board that tracks hits and misses (using `X` and `O`), and a playing board that tracks ship positions (using ship letters). Mixing up the two kinds of boards could be catastrophic.

The player and enemy tracking and playing boards now all use `Grid`, and the use of methods like `Grid.display()` prevents us from passing the wrong board to the display method. However, functions like `display_overlay(targetting: Grid, playing: Grid)` which take two grids may still have their arguments accidentally swapped. Furthermore, if we bundle methods like `place_ship()` and `update_tracking()` into the `Grid` class, we can accidentally call them on the wrong board. For example, if we call `place_ship()` on a tracking board, it will not work as expected.

This is a problem of **type safety**. We want to ensure that the right methods are called on the right objects. In Python, we can use **subclassing** to create new classes that inherit from existing classes. This allows us to create specialized versions of a class while still using the same public interface.

In this lesson, we will create `TrackingBoard` and `PlayingBoard` subclasses. These two classes bundle additional data and methods specific to their purpose, yet can still be used wherever `Grid` is used.

In [ ]:
# Run this cell to make the Grid class available for subsequent cells.
from starting_code import GRID_SIZE, HORIZONTAL, VERTICAL, EMPTY, HIT, MISS

class Grid:
    """Represents a grid in Battleships."""
    def __init__(self, grid_size, char: str):
        self._grid = [char] * (grid_size * grid_size)
        self.grid_size = grid_size

    def get(self, x: int, y: int) -> str:
        """Get the character at the specified coordinates."""
        return self._grid[x * self.grid_size + y]
    
    def set(self, x: int, y: int, char: str) -> None:
        """Set the character at the specified coordinates."""
        self._grid[x * self.grid_size + y] = char

    def is_unoccupied(self, row: int, col: int, size: int, orientation: str) -> bool:
        if orientation == HORIZONTAL:
            for i in range(size):
                if self.get(row, col + i) != EMPTY:
                    return False
        elif orientation == VERTICAL:
            for i in range(size):
                if self.get(row + i, col) != EMPTY:
                    return False
        return True
    
    def display(self) -> None:
        """Display the grid."""
        # Column label row
        print("  " + " ".join(str(i) for i in range(GRID_SIZE)))
        # Row label followed by the row contents
        # We can't just iterate over the grid, assuming it is a 2D list,
        # because it might not be.
        for i in range(self.grid_size):
            # Row header
            print(str(i) + " ", end="")
            for j in range(self.grid_size):
                print(self.get(i, j), end=" ")
            print()  # linebreak

## `TrackingBoard` and `PlayingBoard` classes

Looking at our starting code again, we can see `TrackingBoard` which tracks hits and misses will require the following attributes and methods:

- `hits`: `int ` 
  Number of hits scored
- `targetting_update(hit_what: str, x: int, y: int)`  
  Update the targetting board with a hit or miss

The `PlayingBoard` class will require the following attributes and methods:

- `ship_cells`: `dict[str, int]`  
  Number of cells remaining for each ship on the board
- `sunk_ships`: `list[str]`  
  List of sunk ships
- `is_target_hit(x: int, y: int) -> bool`  
  Check if the opponent scored a hit on this board
- `playing_update(x: int, y: int)`  
  Update the playing board with opponent's guesses

Despite presenting a different set of attributes and methods, both classes will still need to be useable wherever `Grid` is used. This means we need to be able to call `get()`, `set()`, and `display()` on both classes, and pass them to functions that take a `Grid` as a parameter.

One way to do this is to copy all the methods from `Grid` into both classes. This is called **code duplication** and is a bad idea. If we need to change the implementation of `display()`, we would have to change it correctly in three places.

Wouldn't it be nice if `TrackingBoard` and `PlayingBoard` could just use those methods from `Grid`, without having to copy them? This is where **inheritance** comes in.

## Inheritance

**Inheritance** is a property whereby a **child class** (or **subclass**) is able to use the **public** methods and attributes of a **parent class** (or **superclass**). In our case, `TrackingBoard` and `PlayingBoard` will inherit from `Grid`. This means they will have access to all the methods and attributes of `Grid`, including `display()`, `get()`, and `set()`.

(**Note:** the inheritance relationship is one-way; `Grid` is unable to use the methods and attributes of `TrackingBoard` or `PlayingBoard`.)

In Python, we apply inheritance using parentheses in the class definition. For example, to create a `TrackingBoard` class that inherits from `Grid`, we would write:

In [ ]:
class TrackingBoard(Grid):
    """Tracks hits and misses on a grid."""

board = TrackingBoard(10, EMPTY)
board.display()

Notice that even without implementing any new methods, `TrackingBoard` already has access to all the methods and attributes of `Grid`. This is because `TrackingBoard` is a subclass of `Grid`, and therefore inherits all its methods and attributes.

Child classes are also recognised as instances of their parent class. This means that if we have a function that takes a `Grid` as a parameter, we can also pass in a `TrackingBoard`. For exact type matches, use `type(variable) == Class` instead.


In [ ]:
print(isinstance(board, TrackingBoard))  # Should print True
print(isinstance(board, Grid))  # Should print True
print(type(board) == TrackingBoard)  # Should print True
print(type(board) == Grid)  # Should print False


To implement additional methods and attributes, we simply define them in the `TrackingBoard` class. For example, to implement the `targetting_update()` method, we would write:

In [ ]:
class TrackingBoard(Grid):
    """Tracks hits and misses on a grid."""
    
    def update(self, hit_what: str, x: int, y: int) -> None:  # renamed from targetting_update()
        """Update the grid with a hit or miss."""
        if hit_what == EMPTY:
            self.set(x, y, MISS)
        else:
            self.set(x, y, HIT)

board = TrackingBoard(10, EMPTY)
board.update('B', 2, 3)
board.display()

### Method overriding and extending

We also need to implement the `__init__()` method. `Grid.__init__()` only sets the `_grid` and `grid_size` attributes, but we also need to set `hits = 0` for `TrackingBoard`. We could do:

```python
class TrackingBoard(Grid):
    def __init__(self, grid_size: int, char: str):
        self._grid = create_grid(grid_size, char)
        self.grid_size = grid_size
        self.hits = 0
```

This would be code duplication, which we are trying to avoid, because it makes refactoring trickier. What if we could first call `Grid.__init__()` to set the `_grid` and `grid_size` attributes, *and then* set `hits = 0`? This is where the `super()` function comes in.

The `super()` function returns a temporary object of the superclass that allows us to call its methods. This means we can call `Grid.__init__()` from within `TrackingBoard.__init__()` to set the `_grid` and `grid_size` attributes, and then set `hits = 0`. The code would look like this:

In [ ]:
class TrackingBoard(Grid):
    """Tracks hits and misses on a grid."""
    def __init__(self, grid_size: int, char: str):
        """Invoke the parent constructor to set grid_size and _grid."""
        super().__init__(grid_size, char)
        # Then continue to initialise additional attributes.
        self.hits = 0

    def update(self, hit_what: str, x: int, y: int) -> None:
        """Update the grid with a hit or miss.
        
        Args:
            hit_what (str): The character representing the hit or miss.
            x (int): The x-coordinate of the hit or miss.
            y (int): The y-coordinate of the hit or miss.
        """
        # We expect hit_what to be either EMPTY or a ship character.
        # If it is EMPTY, we have a miss.
        if hit_what == EMPTY:
            self.set(x, y, MISS)
        else:
            self.set(x, y, HIT)

board = TrackingBoard(10, EMPTY)
print("grid_size:", board.grid_size)
print("hits:", board.hits)

Note that by defining `__init__()` in `TrackingBoard`, we are **overriding** the `__init__()` method of `Grid`. This means that when we create a `TrackingBoard` object (with `TrackingBoard(10, EMPTY)`), it will use the `__init__()` method of `TrackingBoard`, not the one from `Grid`. If we did not call the `__init__()` method of `Grid` using `super()`, it would not be invoked.

Using `super().__init__()` allows us to call the `__init__()` method of the superclass, which sets the `_grid` and `grid_size` attributes. We can then set any additional attributes we need in the child class. This pattern is called **method extension**. It allows us to extend the functionality of a method in a child class while still using the implementation from the parent class.

## Exercise 1

Implement the `PlayingBoard` class. It should inherit from `Grid`, and implement the following methods:
- `__init__(self, grid_size: int, char: str)`
  Call `Grid.__init__()` to set the `_grid` and `grid_size` attributes, and set `ship_cells = {}` and `sunk_ships = []`.
- `is_target_hit(self, x: int, y: int) -> bool`
  Check if the opponent scored a hit on this board. If the cell is not empty, return `True`. Otherwise, return `False`.
- `update(self, x: int, y: int)` (renamed from `playing_update`)
  Update the playing board with opponent's guesses. If the cell is not empty, remove the ship from `ship_cells` and add it to `sunk_ships`. Otherwise, do nothing.

You may edit the starting code to implement the above. Remember to obey the public interface of the `Grid` class.

In [ ]:
# Write your code for Exercise 1 here.


## Exercise 2

Extend the `display()` method of the `Grid` parent class.

- `TrackingBoard.display()` should display `Hits: <n>` at the top of the board, where `<n>` is the number of hits scored
- `PlayingBoard.display()` should display `Ships sunk: <...>` at the bottom of the board, where `<...>` are letters of ships sunk.

You can use `super().display()` to call the `display()` method of the parent class.

In [ ]:
# Write your code for Exercise 2 here.

# Summary

Research shows that **active recall**, the mental effort of attempting to remember, helps strengthen neuron connections. For each of the questions below, try to recall what you learnt from this lesson before you click to reveal.

<ol>

<li><details>
    <summary>What is <b>inheritance</b>? (click to reveal)</summary>
    <p>Inheritance refers to the ability of child classes to access <i>public</i> methods of the parent class.</p>
</details></li>

<li><details>
    <summary>Why can't child classes access public attributes of a parent class? (click to reveal)</summary>
    <p>Attributes are bound to objects, not to classes. The notion of inheritance applies to classes, not objects.</p>
</details></li>

<li><details>
    <summary>How does inheritance promote code reuse? (click to reveal)</summary>
    <p>Inheritance allows child classes to access parent class methods instead of reimplementing them. They may also extend those methods. This allows code written in the parent class to be written once and invoked wherever it is needed.</p>
</details></li>